# 08 — Reconciliação e decisão de construção

Execute após 03–07. Este notebook simula bronze → silver → contagens apenas para a amostra. `PASSOU` é evidência restrita à amostra e não prova completude histórica. A decisão por funcionalidade usa checks obrigatórios explícitos; checks inconclusivos mantêm a funcionalidade pendente ou bloqueada.

In [1]:
from pathlib import Path
import sys
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'validation_utils.py').exists())
sys.path.insert(0, str(root / 'scripts'))
from validation_utils import (Probe, OUT, CHECKS, MANIFEST, MANIFEST_FIELDS, api_camara, api_senado,
                              record, manifest_rows, save_manifest, rows_camara, count_values, csv_head)
pd.set_option('display.max_colwidth', 90)

import csv, json
from collections import Counter
probe=Probe('08',max_calls=70)
manifest=manifest_rows()
assert manifest, 'Execute 03 primeiro.'

## Respostas brutas → linhas tipadas → agregação sem multiplicação

In [2]:
projects=[x for x in manifest if x['entidade']=='projeto']
raw=[]; failures=[]
for item in projects:
    r=probe.json(item['url'])
    if not r.ok: failures.append({'id':item['id'],'erro':r.error}); continue
    d=r.data.get('dados',{}) if item['fonte']=='camara' else r.data
    raw.append({'fonte':item['fonte'],'id':str(d.get('id','')),'objeto':d,'url':r.url,'horario':r.collected_at})
silver=[]
for item in raw:
    d=item['objeto']
    silver.append({'fonte':item['fonte'],'id':item['id'],
                   'tipo':d.get('siglaTipo') if item['fonte']=='camara' else d.get('sigla'),
                   'data_apresentacao':d.get('dataApresentacao') if item['fonte']=='camara' else (d.get('documento') or {}).get('dataApresentacao')})
frame=pd.DataFrame(silver)
if not frame.empty:
    frame['ano']=frame['data_apresentacao'].fillna('').str[:4]
    gold=frame.drop_duplicates(['fonte','id']).groupby(['fonte','ano','tipo'],dropna=False).size().rename('projetos_unicos').reset_index()
else: gold=pd.DataFrame()
display(frame.head(18))
display(gold)
duplicate=int(frame.duplicated(['fonte','id']).sum()) if not frame.empty else 0
counts={'solicitados':len(projects),'bronze':len(raw),'silver':len(frame),'gold_total':int(gold['projetos_unicos'].sum()) if not gold.empty else 0,
        'IDs_duplicados':duplicate,'falhas':len(failures),
        'ano_apresentacao_divergente_manifesto':sum((item['data_apresentacao'] or '')[:4]!=next((p['ano'] for p in projects if p['fonte']==item['fonte'] and p['id']==item['id']),'') for item in silver)}
record('08','reconciliacao_projetos','ambas','detalhes do manifesto',len(raw),
       'Linhas brutas e normalizadas preservam projetos únicos sem multiplicação no agregado',counts,
       'PASSOU' if not failures and duplicate==0 and counts['ano_apresentacao_divergente_manifesto']==0 and counts['bronze']==counts['silver']==counts['gold_total'] else 'INCONCLUSIVO',str(failures[:3]))

,fonte,id,tipo,data_apresentacao,ano
0,camara,2647384,PL,2026-10-08T17:21,2026
1,camara,2647345,PL,2026-10-07T16:07,2026
2,camara,2599994,PLP,2026-02-02T09:22,2026
3,camara,2600175,PLP,2026-02-02T15:05,2026
4,camara,2604173,PEC,2026-02-24T15:48,2026
5,camara,2623400,PEC,2026-05-11T14:49,2026
6,senado,9101538,PL,2026-09-01,2026
7,senado,9101917,PL,2026-09-01,2026
8,senado,9104876,PLP,2026-09-02,2026
9,senado,9105948,PLP,2026-09-03,2026


,fonte,ano,tipo,projetos_unicos
0,camara,2023,PEC,1
1,camara,2023,PL,1
2,camara,2023,PLP,1
3,camara,2024,PEC,1
4,camara,2024,PL,1
5,camara,2024,PLP,1
6,camara,2025,PEC,1
7,camara,2025,PL,1
8,camara,2025,PLP,1
9,camara,2026,PEC,2


{'notebook': '08',
 'check': 'reconciliacao_projetos',
 'fonte': 'ambas',
 'url_parametros': 'detalhes do manifesto',
 'coletado_em_utc': '2026-10-09T14:24:49+00:00',
 'amostra_n': 30,
 'regra': 'Linhas brutas e normalizadas preservam projetos únicos sem multiplicação no agregado',
 'contagens': '{"IDs_duplicados": 0, "ano_apresentacao_divergente_manifesto": 0, "bronze": 30, "falhas": 0, "gold_total": 30, "silver": 30, "solicitados": 30}',
 'estado': 'PASSOU',
 'motivo': '[]'}

## Paginação e incerteza de cobertura

In [3]:
page1=probe.json(api_camara('proposicoes'),{'ano':2026,'siglaTipo':'PL','dataApresentacaoInicio':'2026-07-01',
          'dataApresentacaoFim':'2026-09-30','itens':5,'pagina':1,'ordem':'ASC','ordenarPor':'id'})
page2=probe.json(api_camara('proposicoes'),{'ano':2026,'siglaTipo':'PL','dataApresentacaoInicio':'2026-07-01',
          'dataApresentacaoFim':'2026-09-30','itens':5,'pagina':2,'ordem':'ASC','ordenarPor':'id'})
ids1=[str(x.get('id')) for x in rows_camara(page1.data)] if page1.ok else []
ids2=[str(x.get('id')) for x in rows_camara(page2.data)] if page2.ok else []
overlap=len(set(ids1)&set(ids2))
wrong_year=sum(str(x.get('dataApresentacao',''))[:4]!='2026' for x in rows_camara(page1.data)+rows_camara(page2.data))
display(pd.DataFrame([{'pagina':1,'linhas':len(ids1),'IDs':ids1},{'pagina':2,'linhas':len(ids2),'IDs':ids2}]))
record('08','paginacao_camara','camara',f'{page1.url} | {page2.url}',len(ids1)+len(ids2),
       'Páginas adjacentes não repetem IDs e respeitam data de apresentação',{'pagina1':len(ids1),'pagina2':len(ids2),'sobreposicao':overlap,'ano_incoerente':wrong_year},
       'PASSOU' if page1.ok and page2.ok and ids1 and ids2 and overlap==0 and wrong_year==0 else 'INCONCLUSIVO',
       'Páginas podem mudar durante atualização da fonte; amostra não prova snapshot estável')

,pagina,linhas,IDs
0,1,5,"[2635996, 2636035, 2636038, 2636044, 2636046]"
1,2,5,"[2636048, 2636049, 2636052, 2636054, 2636055]"


{'notebook': '08',
 'check': 'paginacao_camara',
 'fonte': 'camara',
 'url_parametros': 'https://dadosabertos.camara.leg.br/api/v2/proposicoes?ano=2026&siglaTipo=PL&dataApresentacaoInicio=2026-07-01&dataApresentacaoFim=2026-09-30&itens=5&pagina=1&ordem=ASC&ordenarPor=id | https://dadosabertos.camara.leg.br/api/v2/proposicoes?ano=2026&siglaTipo=PL&dataApresentacaoInicio=2026-07-01&dataApresentacaoFim=2026-09-30&itens=5&pagina=2&ordem=ASC&ordenarPor=id',
 'coletado_em_utc': '2026-10-09T14:24:52+00:00',
 'amostra_n': 10,
 'regra': 'Páginas adjacentes não repetem IDs e respeitam data de apresentação',
 'contagens': '{"ano_incoerente": 0, "pagina1": 5, "pagina2": 5, "sobreposicao": 0}',
 'estado': 'PASSOU',
 'motivo': 'Páginas podem mudar durante atualização da fonte; amostra não prova snapshot estável'}

## Matriz de decisão e relatório pequeno

In [4]:
with CHECKS.open(encoding='utf-8-sig',newline='') as f: checks=list(csv.DictReader(f))
latest={(x['notebook'],x['check']):x for x in checks}
requirements={
 'composição atual': [('03','contrato_camara'),('03','contrato_senado'),('04','composicao_atual')],
 'exploração de projetos': [('03','estratos_manifesto'),('04','ids_projeto'),('08','reconciliacao_projetos')],
 'presença Câmara': [('05','presenca_camara')],
 'presença Senado': [('05','presenca_senado')],
 'participação em votações': [('05','votos_camara'),('05','codigos_voto_senado')],
 'aprovação e norma': [('06','aprovacao_votacao'),('06','conclusao_casa'),('06','norma_publicada')],
 'classificação ideológica': [('07','ideologia_atual')],
 'direção de medida': [('07','temas_e_rotulos')]}
blocked={'presença Senado','classificação ideológica','direção de medida'}
decisions=[]
for feature,keys in requirements.items():
    states=[latest.get(k,{}).get('estado','AUSENTE') for k in keys]
    state='apto para iniciar' if all(x=='PASSOU' for x in states) else ('bloqueado' if feature in blocked else 'pendente')
    decisions.append({'funcionalidade':feature,'decisão':state,'checks':', '.join(f'{a}/{b}: {latest.get((a,b),{}).get("estado","AUSENTE")}' for a,b in keys)})
display(pd.DataFrame(decisions))
report=OUT/'resumo.md'
lines=['# Validação exploratória — relatório de decisão','',
       'Este relatório reflete somente a amostra. PASSOU não prova completude histórica.','',
       '## Funcionalidades','',
       '| Funcionalidade | Decisão | Evidência |','| --- | --- | --- |']
for x in decisions: lines.append(f"| {x['funcionalidade']} | {x['decisão']} | {x['checks']} |")
lines += ['', '## Checks', '', '| Notebook/check | Estado | Amostra | Motivo |','| --- | --- | ---: | --- |']
for x in sorted(checks,key=lambda r:(r['notebook'],r['check'])):
    reason=x['motivo'].replace('|','/')[:180]
    lines.append(f"| {x['notebook']}/{x['check']} | {x['estado']} | {x['amostra_n']} | {reason} |")
lines += ['', 'Pendências não autorizam taxa, ranking ou rótulo público. Revisar docs/especificacao_funcional.md antes da implementação.']
report.write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('Relatório:',report,'| checks:',len(checks),'| chamadas deste notebook:',probe.calls)

,funcionalidade,decisão,checks
0,composição atual,apto para iniciar,"03/contrato_camara: PASSOU, 03/contrato_senado: PASSOU, 04/composicao_atual: PASSOU"
1,exploração de projetos,apto para iniciar,"03/estratos_manifesto: PASSOU, 04/ids_projeto: PASSOU, 08/reconciliacao_projetos: PASSOU"
2,presença Câmara,pendente,05/presenca_camara: INCONCLUSIVO
3,presença Senado,bloqueado,05/presenca_senado: INCONCLUSIVO
4,participação em votações,pendente,"05/votos_camara: INCONCLUSIVO, 05/codigos_voto_senado: INCONCLUSIVO"
5,aprovação e norma,pendente,"06/aprovacao_votacao: INCONCLUSIVO, 06/conclusao_casa: INCONCLUSIVO, 06/norma_publicad..."
6,classificação ideológica,bloqueado,07/ideologia_atual: INCONCLUSIVO
7,direção de medida,bloqueado,07/temas_e_rotulos: INCONCLUSIVO


Relatório: G:\Meu Drive\Projetos pessoais\Dashboard Lesgislativo BR\docs\validacao\resumo.md | checks: 29 | chamadas deste notebook: 32
